In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

from core.logging import configure_logging
from core.store import PostgresStore
from core.runs import (
    RunRepository,
    RunTracker
)

from registry import RegistryClient

from gl.models import GLSegments
from gl import GLClient
from gl.repository import GLRepository

from recon import ReconClient

from break_analysis.tools import RegistryTools
from break_analysis import BreakAnalysisAgent
from break_analysis.models import BreakRecord
from break_analysis.builder import BreakCaseBuilder


configure_logging()

def display_df(df):
    display(df.toPandas())

In [3]:
spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('break-agent-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/12 10:31:52 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/09/12 10:31:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8c9b3a4c-7800-4304-a91c-777ccdbd32e1;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 83ms :: artifacts dl 3ms
	:: modules in use:
	org.checkerframework#

In [4]:
registry_client = RegistryClient.from_db(
    spark=spark,
    entity_table='registry.gl_entity',
    department_table='registry.gl_dept',
    branch_table='registry.gl_branch',
    account_table='registry.gl_account',
    sub_account_table='registry.gl_sub_account',
    affiliate_table='registry.gl_affiliate',
    product_table='registry.gl_product',
    book_table='registry.gl_book',
    source_table='registry.gl_source',
)

run_repository = RunRepository()
run_tracker = RunTracker(
    repository = run_repository
)

gl_store = PostgresStore(
    spark,
    table_names={
        'SEGMENT_DEFAULT': 'gl.segment_default',
        'POSTING': 'gl.posting',
        'REJECTION': 'gl.rejection',
        'INTERFACE_TRIAL_BALANCE': 'interface.trial_balance',
    },
)

gl_repository = GLRepository(gl_store, spark)
gl = GLClient(gl_repository, registry_client)

recon = ReconClient.from_db(
    spark=spark,
    run_tracker=run_tracker,
    gl=gl,
)

registry_tools = RegistryTools(registry_client=registry_client)

In [5]:
llm = ChatOllama(
    model='qwen3:14b-q4_K_M',
    temperature=0
)

agent = BreakAnalysisAgent(
    llm=llm,
    registry_tools=registry_tools
)

In [6]:
workflow_run_id = 'b40f3d54-2995-4300-9315-5b25f44c6ccb'

recon_df = recon.get_results(workflow_run_id)
breaks_df = recon_df.filter(F.col('DIFFERENCE_AMOUNT') != 0)

breaks = []
for row in breaks_df.collect():
    segments = GLSegments(
        entity_cd = row['ENTITY_CD'],
        branch_cd = row['BRANCH_CD'],
        dept_cd = row['DEPT_CD'],
        gl_account = row['GL_ACCOUNT'],
        sub_account = row['SUB_ACCOUNT'],
        affiliate_cd = row['AFFILIATE_CD'],
        product_cd = row['PRODUCT_CD'],
        book_cd = row['BOOK_CD'],
        source_cd = row['SOURCE_CD'],
    )
    breaks.append(
        BreakRecord(
            recon_result_id= row['RECON_RESULT_ID'],
            workflow_run_id = row['WORKFLOW_RUN_ID'],
            as_of_date = row['AS_OF_DATE'],
            segments = segments,
            accounted_currency = row['ACCOUNTED_CURRENCY'],
            interface_balance = row['INTERFACE_BALANCE'],
            gl_balance = row['GL_BALANCE'],
            difference_amount = row['DIFFERENCE_AMOUNT']
        )
    )

segment_defaults = gl.get_segment_defaults()
case_builder = BreakCaseBuilder(segment_defaults)

break_cases = case_builder.build(breaks)

print(len(breaks), len(break_cases))

2026-09-12 10:31:57,734 | INFO | break_analysis.builder | Building break cases | records=5
2026-09-12 10:31:57,735 | INFO | break_analysis.builder | Break cases built | total=2 | one_to_one=1 | many_to_one=1 | ambiguous=0 | interface_only=0 | gl_only=0 | unmatched=0 | duration_ms=1
5 2


In [7]:
break_case = break_cases[0]

agent.analyze(break_case)

2026-09-12 10:31:57,739 | INFO | break_analysis.agent | Analyzing break case | case_id=e92f07c4 | workflow_run_id=b40f3d54 | topology=ONE_TO_ONE | records=2
2026-09-12 10:31:57,740 | INFO | break_analysis.agent | Invoking LLM | case_id=e92f07c4 | round=1
2026-09-12 10:32:40,213 | INFO | break_analysis.agent | LLM invoked | case_id=e92f07c4 | round=1 | tool_calls=1 | input_tokens=1133 | output_tokens=974 | total_tokens=2107 | duration_ms=42473
2026-09-12 10:32:40,214 | INFO | break_analysis.agent | Tool round | case_id=e92f07c4 | round=1 | tool_calls=1
2026-09-12 10:32:40,443 | INFO | break_analysis.agent | Tool invoked | case_id=e92f07c4 | tool=validate_segment | args={"segment_type": "GL_ACCOUNT", "segment_value": "210000", "business_dt": "2026-03-31"} | duration_ms=229
2026-09-12 10:32:40,443 | INFO | break_analysis.agent | Invoking LLM | case_id=e92f07c4 | round=2
2026-09-12 10:32:51,665 | INFO | break_analysis.agent | LLM invoked | case_id=e92f07c4 | round=2 | tool_calls=1 | input_

BreakAnalysisResult(case_id=UUID('e92f07c4-d207-4ae7-96cc-6d3fb9b15f0b'), recon_result_ids=(UUID('92355ca6-b6d1-4085-9324-bd4ab0bd5369'), UUID('27d6bfb9-89a6-4e60-a96f-85fac68ebfa6')), status=<BreakAnalysisStatus.EXPLAINED: 'EXPLAINED'>, root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, explanation="The 'gl_account' segment with value '210000' exists in the Registry but is marked as inactive (status='I') for the business date 2026-03-31.")

In [8]:
break_case = break_cases[1]

agent.analyze(break_case)

2026-09-12 10:33:37,366 | INFO | break_analysis.agent | Analyzing break case | case_id=b00595c7 | workflow_run_id=b40f3d54 | topology=MANY_TO_ONE | records=3
2026-09-12 10:33:37,367 | INFO | break_analysis.agent | Invoking LLM | case_id=b00595c7 | round=1
2026-09-12 10:34:29,141 | INFO | break_analysis.agent | LLM invoked | case_id=b00595c7 | round=1 | tool_calls=4 | input_tokens=1393 | output_tokens=1209 | total_tokens=2602 | duration_ms=51774
2026-09-12 10:34:29,142 | INFO | break_analysis.agent | Tool round | case_id=b00595c7 | round=1 | tool_calls=4
2026-09-12 10:34:29,258 | INFO | break_analysis.agent | Tool invoked | case_id=b00595c7 | tool=validate_segment | args={"segment_type": "GL_ACCOUNT", "segment_value": "310000", "business_dt": "2026-03-31"} | duration_ms=116
2026-09-12 10:34:29,382 | INFO | break_analysis.agent | Tool invoked | case_id=b00595c7 | tool=validate_segment | args={"segment_type": "GL_ACCOUNT", "segment_value": "4100000", "business_dt": "2026-03-31"} | duratio

BreakAnalysisResult(case_id=UUID('b00595c7-99c2-4ecd-8836-1d6624093fa7'), recon_result_ids=(UUID('b12e2a05-f053-47c1-b217-0b732bcde170'), UUID('095304d2-956f-474f-a9ad-c78472138449'), UUID('946d2aa6-b41b-4094-9437-99529fbfb6f4')), status=<BreakAnalysisStatus.EXPLAINED: 'EXPLAINED'>, root_cause=<RootCause.REGISTRY_INVALID_SEGMENT: 'REGISTRY_INVALID_SEGMENT'>, explanation="Invalid segment values found in investigation records: gl_account '310000' (inactive), sub_account '003000' (inactive), and gl_account '4100000' (missing from Registry).")

In [9]:
# spark.stop()